In [ ]:
# Build regression dataset from 3 CSVs (investors, startups, interactions)
# Run this cell first if investor_startup_matching_regression.csv does not exist.
import pandas as pd
import os
os.chdir(os.path.dirname(os.path.abspath(".")) if "__file__" in dir() else ".")

investors = pd.read_csv("investors.csv")
startups = pd.read_csv("startups.csv")
interactions = pd.read_csv("interactions.csv")
score_map = {"viewed": 45, "shortlisted": 70, "contacted": 90}
interactions["match_score"] = interactions["interaction_type"].map(score_map)
merged = interactions.merge(startups, on="startup_id", how="inner").merge(investors, on="investor_id", how="inner")
df_reg = merged[["industry", "stage", "funding_required_lakhs", "preferred_industry", "preferred_stage",
                 "ticket_min_lakhs", "ticket_max_lakhs", "description", "thesis", "match_score"]].copy()
df_reg.columns = ["startup_industry", "startup_stage", "funding_required_lakhs", "investor_pref_industry",
                  "investor_pref_stage", "investor_ticket_min_lakhs", "investor_ticket_max_lakhs",
                  "startup_desc", "investor_desc", "match_score"]
df_reg.to_csv("investor_startup_matching_regression.csv", index=False)
print("Built investor_startup_matching_regression.csv with", len(df_reg), "rows")

In [1]:
# ================================
# 1. IMPORTS
# ================================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor

# ================================
# 2. LOAD DATASET
# ================================
df = pd.read_csv("investor_startup_matching_regression.csv")

# ================================
# 3. DROP LOCATION (IF PRESENT)
# ================================
df = df.drop(
    ["startup_location", "investor_location_pref"],
    axis=1,
    errors="ignore"
)

# ================================
# 4. SPLIT FEATURES & TARGET
# ================================
X = df.drop("match_score", axis=1)
y = df["match_score"]

# ================================
# 5. NLP → IDEA SIMILARITY
# ================================
tfidf = TfidfVectorizer(stop_words="english")

startup_vec = tfidf.fit_transform(X["startup_desc"])
investor_vec = tfidf.transform(X["investor_desc"])

similarity_scores = []

for i in range(len(X)):
    sim = cosine_similarity(startup_vec[i], investor_vec[i])[0][0]
    similarity_scores.append(sim)

X["idea_similarity"] = similarity_scores

# ================================
# 6. FUNDING FIT FEATURE ⭐⭐⭐⭐⭐
# ================================
X["funding_fit"] = (
    (X["funding_required_lakhs"] >= X["investor_ticket_min_lakhs"]) &
    (X["funding_required_lakhs"] <= X["investor_ticket_max_lakhs"])
)

# ================================
# 7. DROP RAW TEXT
# ================================
X = X.drop(["startup_desc", "investor_desc"], axis=1)

# ================================
# 8. ONE-HOT ENCODE
# ================================
X_encoded = pd.get_dummies(X, drop_first=True)

# Save columns for later scoring
X_encoded_cols = X_encoded.columns

# ================================
# 9. TRAIN / TEST SPLIT
# ================================
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42
)

# ================================
# 10. TRAIN MODEL ⭐ BEST CHOICE
# ================================
model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# ================================
# 11. EVALUATION
# ================================
preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print("MAE:", mae)
print("RMSE:", rmse)

# ================================
# 12. MATCH SCORE GENERATOR ⭐⭐⭐⭐⭐
# ================================
def predict_match_score(startup_data, investor_data):
    
    input_df = pd.DataFrame([{
        "startup_industry": startup_data["industry"],
        "startup_stage": startup_data["stage"],
        "funding_required_lakhs": startup_data["funding_required"],
        
        "investor_pref_industry": investor_data["preferred_industry"],
        "investor_pref_stage": investor_data["preferred_stage"],
        "investor_ticket_min_lakhs": investor_data["ticket_min"],
        "investor_ticket_max_lakhs": investor_data["ticket_max"],
        
        "startup_desc": startup_data["description"],
        "investor_desc": investor_data["description"]
    }])
    
    # --- Idea Similarity ---
    startup_vec = tfidf.transform(input_df["startup_desc"])
    investor_vec = tfidf.transform(input_df["investor_desc"])
    
    similarity = cosine_similarity(startup_vec, investor_vec)[0][0]
    input_df["idea_similarity"] = similarity
    
    # --- Funding Fit ---
    input_df["funding_fit"] = (
        (input_df["funding_required_lakhs"] >= input_df["investor_ticket_min_lakhs"]) &
        (input_df["funding_required_lakhs"] <= input_df["investor_ticket_max_lakhs"])
    )
    
    # Drop text
    input_df = input_df.drop(["startup_desc", "investor_desc"], axis=1)
    
    # Encode
    input_encoded = pd.get_dummies(input_df)
    
    # Align columns ⭐ CRITICAL
    input_aligned = input_encoded.reindex(
        columns=X_encoded_cols,
        fill_value=0
    )
    
    score = model.predict(input_aligned)[0]
    
    return round(float(score), 2)

# ================================
# 13. EXAMPLE TEST
# ================================
startup_example = {
    "industry": "FinTech",
    "stage": "Seed",
    "funding_required": 80,
    "description": "AI-based fraud detection for digital payments"
}

investor_example = {
    "preferred_industry": "FinTech",
    "preferred_stage": "Seed",
    "ticket_min": 50,
    "ticket_max": 200,
    "description": "Investing in FinTech and AI-driven startups"
}

score = predict_match_score(startup_example, investor_example)

print("Predicted Match Score:", score)

MAE: 12.521085632876398
RMSE: 15.468008797039474
Predicted Match Score: 73.29
